# 写真の1点から、ロボットを動かすまで
### Gemini Robotics ER 2 × SuperDex（ロボット・シミュレータ）

AI に写真を見せると「どれを掴めばいいか」を点で指してくれます。
でも、その点だけではロボットは動きません。このノートブックでは、その間を全部つないでみます。

| 手順 | やること | 誰が |
|---|---|---|
| 1 | シミュレータの場面を作って、写真を1枚撮る | シミュレータ（Python 3.12） |
| 2 | 写真を ER 2 に見せて「床のブロックを指して」と聞く | このノートブック（Gemini API） |
| 3 | 写真の上の点（2次元）を、ロボットの座標（3次元）に直す | このノートブック（計算） |
| 4 | 座標をロボットに渡して、掴んで箱へ入れる | シミュレータ（IK ＋ 位置制御） |
| 5 | 床が空になるまで、1〜4 を ER 2 に回させる（監督ループ） | シミュレータの中で ER 2 を呼ぶ |

**進め方**: 上から順にセルを実行します（▶ を押すか Shift+Enter）。最初の準備セルだけ 2〜3 分かかります。

### 先に1つだけ ── API キーをシークレットに登録する

手順2で Gemini の API キーを使います。キーはコードに書かず、Colab の**シークレット**（環境変数のようなもの）に入れておきます。

1. [Google AI Studio](https://aistudio.google.com/) で API キーを発行する（無料枠で足ります）
2. この画面の**左のメニューにある 🔑（シークレット）**を開く
3. 「新しいシークレットを追加」→ 名前 `GEMINI_API_KEY`、値にキーを貼る
4. 「ノートブックからのアクセス」のスイッチを **ON** にする

登録していないと手順2で止まります（そのときはここに戻って登録してから、そのセルだけ実行し直せば続けられます）。

## 準備 ── SuperDex を入れる（2〜3 分）

Colab の Python は 3.13 ですが、SuperDex は 3.12 でしか動きません。
そこで Python 3.12 を追加して、専用の環境（venv）を作ります。
画面が無いので、描画は EGL という仕組みで裏で行います。

In [ ]:
%%bash
set -e
if [ ! -f /content/py.env ]; then
  echo "[1/4] Python 3.12"
  apt-get update -q >/dev/null 2>&1 || true
  apt-get install -y -q python3.12 python3.12-venv python3.12-dev >/dev/null 2>&1 || \
    (add-apt-repository -y ppa:deadsnakes/ppa >/dev/null 2>&1 && apt-get update -q >/dev/null 2>&1 && \
     apt-get install -y -q python3.12 python3.12-venv python3.12-dev >/dev/null 2>&1)
  python3.12 --version
  python3.12 -m venv /content/venv
  echo "PY=/content/venv/bin/python" > /content/py.env
fi
source /content/py.env
echo "[2/4] SuperDex"
$PY -m pip install -q --upgrade pip >/dev/null
$PY -m pip install -q superdex pillow numpy 2>&1 | tail -2
echo "[3/4] 画面なし描画（EGL）"
apt-get install -y -q libegl1 libgles2 libgl1-mesa-dri libglvnd0 >/dev/null 2>&1 || true
echo "[4/4] アセット（3D モデル）"
cd /content
[ -d project_superdex ] || git clone -q --depth 1 --branch stable https://github.com/facebookresearch/project_superdex.git
$PY -c "import superdex.physics, polyscope; print('準備完了:', '$PY')"

### シミュレータ側のスクリプトを置く

次の3つのセルは、シミュレータを動かす Python ファイルをこのランタイムに書き出します（中身は読まなくても進められます）。

- `sim_common.py` ── 場面を組み立てる・カメラの情報を出す・描画する
- `sim_scene.py` ── 場面を作って写真を1枚撮る（手順1）
- `sim_pick.py` ── 指定したブロックを掴んで指定した場所へ置く（手順4）
- `sim_loop.py` ── 床が空になるまで ER 2 に聞きながら回す（手順5）

In [ ]:
%%writefile /content/sim_common.py
"""シミュレータ側の共通部品 ── 場面を組み立てる・カメラの情報を出す・描画する・アームを動かす。

Colab 教材（superdex_er2_colab.ipynb）から %%writefile で /content に書き出して使う。
ノートブック本体（Python 3.13）からは import できないので、sim_scene.py / sim_pick.py / sim_loop.py を
venv の Python 3.12 で実行し、ファイル（PNG / JSON）で受け渡す。
"""
import json
import numpy as np
from PIL import Image
import polyscope as ps
import superdex.physics as physics
import superdex.robotics as robotics
from superdex.physics.paths import resolve_asset, resolve_asset_root
from superdex.physics.utils.scene_helpers import find_actor
from superdex.physics.viewer import Viewer, ViewerCfg

# ---- 部品（SuperDex 同梱のアセット） ----
BOT = "bots/arm_hand_combos/fr3_v2_2f_85/fr3_v2_2f_85.superdex_bot"   # FR3 アーム + 2F-85 グリッパ
BLOCKS = "prefabs/box_and_blocks/block_{}.mochi_prefab"                # 25mm のブロック
BOX = "prefabs/box_and_blocks/box_and_blocks.mochi_prefab"             # 仕切り付きの箱

# ---- 配置（単位 m・Z が上） ----
# ⚠ prefab 名の色と、描画で見える色は一致しない（red が紫に見える等）。決定は座標で行う。
LAYOUT = [("red", [0.45, 0.10, 0.03]), ("green", [0.45, -0.06, 0.03]),
          ("blue", [0.57, 0.04, 0.03]), ("yellow", [0.57, -0.14, 0.03])]
BOX_POSE = [0.40, 0.38, 0.0]          # 箱の「手前の角」（中心ではない）
LOOK_FROM, LOOK_AT = [1.62, -1.22, 1.15], [0.52, 0.18, 0.16]   # カメラの位置と注視点
W, H = 1280, 720                       # 描画サイズ（実際のバッファはこれより大きいことがある）
DT = 1.0 / 200.0                       # 物理の刻み（秒）
Z_BLOCK = 0.0133                       # 静定後のブロック重心の高さ（逆投影に使う平面）

# ---- アームの動かし方 ----
EE_LINK = "fr3_link8"
KNUCKLES = ("2f_85_left_knuckle_joint", "2f_85_right_knuckle_joint")
EE_DOWN = [np.pi, 0.0, 0.0]            # 手先を真下に向ける
GRIP_CLOSE = 0.62                      # グリッパを閉じる角度（rad）
LIFT = 0.18                            # 掴んだあと持ち上げる高さ（m）
DROP_Z = 0.34                          # 箱の壁（0.16m）を越えて離す高さ（m）
HOME = [0.40, 0.0, 0.45]               # 置いたあとに戻る位置
REACH = 0.80                           # 楽に届く距離（根元から・m）。FR3 の最大は約 0.85 m
NEAR_SLOT = [0.54, 0.54]               # 箱の、ロボットに近い側の区画の中ほど


def make_bot(scene, ctx):
    bp = robotics.load_bot_prefab_from_file(str(resolve_asset(BOT)))
    for i in range(len(bp.links)):
        bp.links[i].has_gravity = False   # 簡易重力補償
    return robotics.create_bot(scene, bp, ctx), bp


def build_scene(name="scene"):
    """床・箱・ブロック4個・ロボットを置いた場面を作る。"""
    physics.initialize(num_worker_threads=0)
    scene = physics.create_scene(name)
    scene.set_gravity([0, 0, -9.81])
    scene.create_rigid_actor(name="ground",
                             shape=physics.create_plane_shape(normal=[0, 0, 1], distance=0.0),
                             is_static=True)
    physics.prefab.add_to_scene(prefab_path=str(resolve_asset(BOX)), root_path=str(resolve_asset_root(BOX)),
                                scene=scene, params=physics.prefab.PrefabParams(name="box", translation=BOX_POSE))
    for color, pos in LAYOUT:
        rel = BLOCKS.format(color)
        physics.prefab.add_to_scene(prefab_path=str(resolve_asset(rel)), root_path=str(resolve_asset_root(rel)),
                                    scene=scene, params=physics.prefab.PrefabParams(name=f"blk_{color}", translation=pos))
    ctx = robotics.create_context()
    bot, bp = make_bot(scene, ctx)
    return scene, ctx, bot, bp


def settle(scene, seconds):
    """物理を進めて、置いたブロックが床に落ち着くのを待つ。"""
    for _ in range(int(seconds / DT)):
        scene.step(DT)


def make_viewer(scene):
    viewer = Viewer(ViewerCfg(offscreen=True, size=(W, H), coordinate_system="ros"))
    viewer.set_scene(scene)
    viewer.set_camera_view(look_from=LOOK_FROM, look_at=LOOK_AT)
    return viewer


def render(viewer):
    """1枚描画して PIL Image で返す。"""
    return Image.fromarray(np.asarray(viewer.render())[..., :3])


class Camera:
    """写真の点を 3 次元に戻すのに必要なカメラの情報（位置・向き・画角・縦横比）。"""

    def __init__(self, viewer, img):
        self.pos = np.asarray(viewer.get_camera_position(), float)
        f = np.asarray(viewer.get_camera_look_dir(), float); self.f = f / np.linalg.norm(f)
        up = np.asarray(viewer.get_camera_up_dir(), float)
        r = np.cross(self.f, up); self.r = r / np.linalg.norm(r)
        self.u = np.cross(self.r, self.f)
        self.fov_deg = float(ps.get_vertical_fov_degrees())
        self.aspect = img.size[0] / img.size[1]
        self.image_size = list(img.size)

    def as_dict(self):
        return {"pos": self.pos.tolist(), "f": self.f.tolist(), "r": self.r.tolist(), "u": self.u.tolist(),
                "fov_deg": self.fov_deg, "aspect": self.aspect, "image_size": self.image_size, "z_block": Z_BLOCK}

    def unproject(self, yx, z0=Z_BLOCK):
        """写真の点 [y, x]（0〜1000）→ 平面 z=z0 との交点（ワールド）。床と交わらなければ None。"""
        y, x = yx
        nx, ny = x / 500 - 1, 1 - y / 500
        tan_half = np.tan(np.deg2rad(self.fov_deg) / 2)
        d = self.f + nx * tan_half * self.aspect * self.r + ny * tan_half * self.u
        d /= np.linalg.norm(d)
        if d[2] > -1e-6:
            return None
        t = (z0 - self.pos[2]) / d[2]
        return self.pos + t * d

    def project(self, P):
        """ワールド座標 → 写真の点 [y, x]（0〜1000）。カメラの後ろなら None。"""
        v = np.asarray(P, float) - self.pos
        xc, yc, zc = v @ self.r, v @ self.u, v @ self.f
        if zc <= 0:
            return None
        tan_half = np.tan(np.deg2rad(self.fov_deg) / 2)
        return [(1 - (yc / zc) / tan_half) / 2 * 1000, ((xc / zc) / (tan_half * self.aspect) + 1) / 2 * 1000]


def camera_params(viewer, img):
    return Camera(viewer, img).as_dict()


def block_actor(scene, color):
    return find_actor(scene, f"blk_{color}/Block_{color}")


def block_xyz(scene, color):
    return np.asarray(block_actor(scene, color).get_center_of_mass_transform().translation, float)


def box_aabb(scene):
    a = find_actor(scene, "box/Box").get_aabb_world()
    return np.asarray(a.min, float), np.asarray(a.max, float)


def in_box(scene, color):
    mn, mx = box_aabb(scene)
    p = block_xyz(scene, color)
    return bool(mn[0] <= p[0] <= mx[0] and mn[1] <= p[1] <= mx[1] and p[2] <= mx[2])


def truth(scene):
    """シミュレータだから分かる「本当の位置」。ブロックの重心と箱の範囲。"""
    mn, mx = box_aabb(scene)
    return {"blocks": {c: block_xyz(scene, c).tolist() for c, _ in LAYOUT},
            "box": {"min": mn.tolist(), "max": mx.tolist()}}


def dump(obj, path):
    with open(path, "w") as fp:
        json.dump(obj, fp, ensure_ascii=False, indent=1)


class ArmControl:
    """IK（分身の場面で関節角を解く）＋ 位置制御（解いた角度を実際のロボットに追わせる）。"""

    def __init__(self, scene, ctx, bot, bp, controller_path):
        actor = bot.get_articulated_actor()
        self.ik_scene = physics.create_scene("ik")
        ik_bot, _ = make_bot(self.ik_scene, ctx)
        self.ik_actor = ik_bot.get_articulated_actor()
        ik_ee = next(h for h in self.ik_actor.get_nested_link_actors()
                     if self.ik_scene.get_actor(h).get_name().endswith("/" + EE_LINK))
        self.ik = physics.experimental.create_ik_solver(self.ik_scene)
        self.ik_pos = self.ik.create_position_target(ik_ee, [0, 0, 0], [0, 0, 0], 1.0e4)
        self.ik.create_rotation_target(ik_ee, [0, 0, 0], EE_DOWN, 1.0e2)
        self.ik_pose = physics.DynamicArrayReal(self.ik_actor.get_num_dofs())

        self.pc = bot.create_controller("MOCHI_ARTICULATED_POSE")
        self.pc.set_params(robotics.ControllerMochiArticulatedPoseParams.load_from_file(controller_path))
        self.pc.initialize(True)
        self.pc_obsv = robotics.ControllerMochiArticulatedPoseObsv()
        self.pc_target = robotics.ControllerMochiArticulatedPoseTarget()
        self.pc_target.world_from_root = actor.get_root_transform()

        dof, j2 = 0, {}
        for i in range(len(bp.joints)):
            j = bp.joints[i]
            if j.type != physics.ArticulatedJointType.REVOLUTE:
                continue
            j2[j.name] = dof; dof += 1
        self.knuckles = [j2[k] for k in KNUCKLES]

        tf = physics.DynamicArrayTransformRT(len(bp.links)); actor.get_articulated_link_transforms(tf)
        names = [bp.links[i].name for i in range(len(bp.links))]
        lp = lambda n: np.asarray(tf[names.index(n)].translation, float)
        # 手先リンクから指先までの長さ。掴む高さの計算に使う
        self.grip_len = float(lp(EE_LINK)[2] - min(lp("2f_85_left_finger_tip_link")[2],
                                                    lp("2f_85_right_finger_tip_link")[2]))

    def command(self, pos, grip):
        """手先を pos へ、グリッパを grip（0=開 1=閉）に。scene.step() の直前に毎回呼ぶ。"""
        self.ik_pos.set_target_position(list(map(float, pos))); self.ik.solve_ik()
        self.ik_actor.get_articulated_pose(self.ik_pose)
        q = np.array(self.ik_pose, float)
        q[self.knuckles[0]] = GRIP_CLOSE * grip; q[self.knuckles[1]] = -GRIP_CLOSE * grip
        self.pc_target.pose_dofs = q.tolist(); self.pc.compute_output(self.pc_obsv, self.pc_target)

    def pick_place_plan(self, P, place_xy):
        """ブロック P を掴んで place_xy へ置き、HOME へ戻る計画。(時刻, 手先の位置, グリッパ) の列。"""
        gz = P[2] + self.grip_len - 0.005
        above = gz + LIFT
        drop = [place_xy[0], place_xy[1], DROP_Z]
        return [(1.4, [P[0], P[1], above], 0.0), (2.6, [P[0], P[1], gz], 0.0),
                (3.6, [P[0], P[1], gz], 1.0), (5.0, [P[0], P[1], above], 1.0),
                (6.8, drop, 1.0), (7.6, drop, 0.0), (8.4, list(HOME), 0.0)]


def waypoint(plan, t):
    """計画の間を、なめらかに補間する。"""
    pp, pg, t0 = np.array(plan[0][1], float), 0.0, 0.0
    for t1, p, g in plan:
        if t <= t1:
            a = np.clip((t - t0) / max(t1 - t0, 1e-6), 0, 1); a = a * a * (3 - 2 * a)
            return pp + a * (np.array(p, float) - pp), pg + a * (g - pg)
        pp, pg, t0 = np.array(p, float), g, t1
    return pp, pg

In [ ]:
%%writefile /content/sim_scene.py
"""場面を作って「写真」を1枚撮る。カメラの情報と本当の位置も書き出す。

  SUPERDEX_ASSETS_PATH=<assets> python sim_scene.py OUTDIR
  → OUTDIR/scene.png  OUTDIR/camera.json  OUTDIR/truth.json
"""
import os, sys
import sim_common as S

out = sys.argv[1] if len(sys.argv) > 1 else "step1"
os.makedirs(out, exist_ok=True)

scene, ctx, bot, bp = S.build_scene("scene")
S.settle(scene, 0.8)                     # ブロックが床に落ち着くまで待つ
viewer = S.make_viewer(scene)
img = S.render(viewer)
img.save(os.path.join(out, "scene.png"))
S.dump(S.camera_params(viewer, img), os.path.join(out, "camera.json"))
S.dump(S.truth(scene), os.path.join(out, "truth.json"))
print("saved:", os.path.join(out, "scene.png"), img.size)

# polyscope は終了時にクラッシュ表示を出すことがあるので、後片付けをせずに抜ける（出力は保存済み）
sys.stdout.flush(); os._exit(0)

In [ ]:
%%writefile /content/sim_pick.py
"""指定したブロックを掴んで、指定した場所へ置く（IK ＋ 位置制御）。

  SUPERDEX_ASSETS_PATH=<assets> python sim_pick.py --target red --place 0.54 0.54 --out step4 \
      [--pick-yx 579 471] [--place-yx 520 610]

  --target   掴むブロック（prefab 名: red / green / blue / yellow）
  --place    置く場所のワールド座標 x y（m）。箱の中を指定する
  --pick-yx / --place-yx  ER 2 が返した画像上の点。動画に丸を描くためだけに使う
  → OUT/f_0000.png …（コマ）と OUT/result.json（箱に入ったか）
"""
import argparse, json, os, sys
import numpy as np
from PIL import ImageDraw
import sim_common as S

ap = argparse.ArgumentParser()
ap.add_argument("--target", required=True)
ap.add_argument("--place", nargs=2, type=float, required=True)
ap.add_argument("--out", default="step4")
ap.add_argument("--pick-yx", nargs=2, type=float)
ap.add_argument("--place-yx", nargs=2, type=float)
ap.add_argument("--controller", default="/content/fr3_v2_2f_85_pose.superdex_controller")
ap.add_argument("--no-render", action="store_true")
args = ap.parse_args()
os.makedirs(args.out, exist_ok=True)

# ---- 場面（sim_scene.py と同じ）と、アームの制御 ----
scene, ctx, bot, bp = S.build_scene("pick")
arm = S.ArmControl(scene, ctx, bot, bp, args.controller)
S.settle(scene, 0.8)

P = S.block_xyz(scene, args.target)          # 掴む位置は「本当の重心」を使う（ER 2 は「どれ」を決めるだけ）
plan = arm.pick_place_plan(P, args.place)
viewer = None if args.no_render else S.make_viewer(scene)
n = int(plan[-1][0] / S.DT); every = max(1, n // 200); saved = 0; zmax = -9.0


def draw_marks(img):
    d = ImageDraw.Draw(img)
    sx, sy = img.size[0] / 1000, img.size[1] / 1000
    for yx, col in ((args.pick_yx, (255, 70, 70)), (args.place_yx, (90, 170, 255))):
        if yx is None:
            continue
        px, py = int(yx[1] * sx), int(yx[0] * sy)
        d.ellipse([px - 30, py - 30, px + 30, py + 30], outline=col, width=6)
    return img


for k in range(n):
    pos, grip = S.waypoint(plan, k * S.DT)
    arm.command(pos, grip)
    scene.step(S.DT)
    zmax = max(zmax, float(S.block_xyz(scene, args.target)[2]))
    if viewer is not None and k % every == 0:
        draw_marks(S.render(viewer)).save(os.path.join(args.out, "f_%04d.png" % saved)); saved += 1

S.settle(scene, 1.0)                          # 離した直後は空中にあるので、落ち着くまで待ってから判定
end = S.block_xyz(scene, args.target)
result = {"target": args.target, "start": P.tolist(), "end": end.tolist(),
          "lifted_mm": round(zmax * 1000), "in_box": S.in_box(scene, args.target), "frames": saved}
S.dump(result, os.path.join(args.out, "result.json"))
print(json.dumps(result, ensure_ascii=False))
sys.stdout.flush(); os._exit(0)

In [ ]:
%%writefile /content/sim_loop.py
"""ER 2 を「監督」にして、床のブロックが全部箱に入るまで回す。

  GEMINI_API_KEY=... SUPERDEX_ASSETS_PATH=<assets> python sim_loop.py --out step5 [--max-rounds 6] [--no-render]

毎周回:
  1. いまの場面を1枚描いて ER 2 に見せる
  2. ER 2 が「床に何個残っているか」「次に掴むブロック」「箱のどこへ置くか」を画像の点で返す
  3. 点をロボットの座標に直し、いちばん近い床のブロックを本当の位置で特定する（ER 2 は「どれ」だけ決める）
  4. 掴んで箱へ入れる（IK ＋ 位置制御）。動画には ER 2 の指示（赤=掴む・青=置く）を重ねる
  5. 本当の位置で「入ったか」を判定し、床が空になるか回数の上限まで繰り返す

失敗しても、次の周回で ER 2 が「まだ床に残っている」と見て指示し直す。これが監督ループ。
ER 2 に聞くのは「どれ・どこへ・いくつ残っているか」だけ。失敗の原因は聞かない（動画に写らないことを聞くと、それらしい答えを作る）。

→ OUT/f_00000.png …（コマ）・OUT/er2_in_<N>.png（ER 2 に見せた画像）・OUT/loop_log.json（周回ごとの記録）
"""
import argparse, base64, json, os, sys, urllib.request
import numpy as np
from PIL import ImageDraw, ImageFont
import sim_common as S

ap = argparse.ArgumentParser()
ap.add_argument("--out", default="step5")
ap.add_argument("--max-rounds", type=int, default=6)       # ブロック 4 個 + 失敗したときの聞き直し分
ap.add_argument("--controller", default="/content/fr3_v2_2f_85_pose.superdex_controller")
ap.add_argument("--no-render", action="store_true")
args = ap.parse_args()
os.makedirs(args.out, exist_ok=True)

API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not API_KEY:
    sys.exit("環境変数 GEMINI_API_KEY がありません（ノートブックはシークレットから渡します）")
ER2_URL = "https://generativelanguage.googleapis.com/v1beta/interactions"

PROMPT = """これはロボットアームと、床に置かれた色つきブロック、黄色い仕切り付きの箱を写したシミュレーション画像です。
タスク: 床に残っているブロックを1個ずつ箱の中へ移すこと。
次の JSON だけを返してください。分からない項目は null にしてください。個数は指定しません。
{
  "remaining_on_floor": <床に残っているブロックの個数。分からなければ null>,
  "pick": [y, x],       // 次に掴むべき床のブロックを指す点
  "pick_color": "<そのブロックの見えたままの色>",
  "place": [y, x],      // 箱の中で、ロボットアームに近い側の区画の、空いている場所を指す点
  "done": <床にブロックが残っていなければ true>
}
座標は [y, x] で 0〜1000 です。"""


def ask_er2(image_path):
    """写真と質問を ER 2 に送り、返ってきた JSON（dict）を返す。読めなければ None。"""
    data = base64.b64encode(open(image_path, "rb").read()).decode()
    body = {"model": "gemini-robotics-er-2-preview",
            "input": [{"type": "image", "data": data, "mime_type": "image/png"},
                      {"type": "text", "text": PROMPT}],
            "generation_config": {"thinking_level": "low"}}
    req = urllib.request.Request(ER2_URL, data=json.dumps(body).encode(),
                                 headers={"x-goog-api-key": API_KEY, "Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            j = json.load(r)
    except urllib.error.HTTPError as e:
        print("  ER 2 エラー:", e.code, e.read()[:300].decode(errors="replace")); return None
    texts = []
    def walk(o):
        if isinstance(o, dict):
            if isinstance(o.get("text"), str): texts.append(o["text"])
            for v in o.values(): walk(v)
        elif isinstance(o, list):
            for v in o: walk(v)
    walk(j)
    text = "\n".join(texts)
    i = text.find("{")
    if i < 0:
        print("  ER 2 の返答に JSON が無い:", text[:200]); return None
    try:
        d, _ = json.JSONDecoder().raw_decode(text[i:])
        return d
    except Exception as e:
        print("  JSON を読めない:", e); return None


def parse_point(v):
    if not isinstance(v, (list, tuple)) or len(v) != 2:
        return None
    try:
        y, x = float(v[0]), float(v[1])
    except (TypeError, ValueError):
        return None
    return [y, x] if (0 <= y <= 1000 and 0 <= x <= 1000) else None


# ---- 場面・カメラ・アーム ----
scene, ctx, bot, bp = S.build_scene("loop")
arm = S.ArmControl(scene, ctx, bot, bp, args.controller)
S.settle(scene, 0.8)
viewer = S.make_viewer(scene)                 # 描画しない設定でも ER 2 に見せる 1 枚は要る
cam = S.Camera(viewer, S.render(viewer))
bmin, bmax = S.box_aabb(scene)

# 動画のラベル用フォント（無ければ既定）
def _font(size):
    for p in ("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "/System/Library/Fonts/Helvetica.ttc"):
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()
FONT, FONT_S = _font(30), _font(24)
RED, BLUE, WHITE, GREEN = (255, 70, 70), (90, 170, 255), (255, 255, 255), (120, 255, 140)
saved = [0]


def draw_frame(marks, lines):
    img = S.render(viewer); d = ImageDraw.Draw(img)
    sx, sy = img.size[0] / 1000, img.size[1] / 1000
    for yx, label, col in marks:
        if yx is None: continue
        px, py = int(yx[1] * sx), int(yx[0] * sy)
        d.ellipse([px - 30, py - 30, px + 30, py + 30], outline=col, width=6)
        d.text((px + 36, py - 14), label, fill=col, font=FONT_S, stroke_width=3, stroke_fill=(0, 0, 0))
    y = 18
    for txt, col in lines:
        d.text((24, y), txt, fill=col, font=FONT, stroke_width=4, stroke_fill=(0, 0, 0)); y += 40
    return img


def save(img):
    img.save(os.path.join(args.out, "f_%05d.png" % saved[0])); saved[0] += 1


def run_motion(P, place_xy, marks, lines):
    plan = arm.pick_place_plan(P, place_xy)
    n = int(plan[-1][0] / S.DT); every = 8
    for k in range(n):
        pos, grip = S.waypoint(plan, k * S.DT)
        arm.command(pos, grip)
        scene.step(S.DT)
        if not args.no_render and k % every == 0:
            save(draw_frame(marks, lines))


# ---- ループ ----
log = []
for rnd in range(1, args.max_rounds + 1):
    floor = [c for c, _ in S.LAYOUT if not S.in_box(scene, c)]
    if not floor:
        break
    still_path = os.path.join(args.out, "er2_in_%d.png" % rnd)
    S.render(viewer).save(still_path)
    ans = ask_er2(still_path)
    entry = {"round": rnd, "floor_truth": len(floor), "er2": ans}
    log.append(entry)
    pick_yx = parse_point(ans.get("pick")) if isinstance(ans, dict) else None
    if pick_yx is None:
        print(f"周回{rnd}: ER 2 が有効な pick を返さず → 聞き直し"); entry["result"] = "no pick"; continue
    pick_w = cam.unproject(pick_yx)
    if pick_w is None:
        print(f"周回{rnd}: pick が床と交わらない → 聞き直し"); entry["result"] = "pick off floor"; continue
    tgt = min(floor, key=lambda c: np.linalg.norm(S.block_xyz(scene, c)[:2] - pick_w[:2]))
    err = float(np.linalg.norm(S.block_xyz(scene, tgt)[:2] - pick_w[:2]))
    if err > 0.10:
        print(f"周回{rnd}: 点が床のどのブロックからも {err*1000:.0f}mm 離れている → 聞き直し"); entry["result"] = "pick too far"; continue
    place_yx = parse_point(ans.get("place"))
    place_w = cam.unproject(place_yx) if place_yx else None
    ok_box = place_w is not None and bmin[0] <= place_w[0] <= bmax[0] and bmin[1] <= place_w[1] <= bmax[1]
    if not ok_box or np.linalg.norm(place_w[:2]) > S.REACH:
        print(f"周回{rnd}: 置き場所が箱の外か遠い → 近い側の区画に寄せる")
        place_w = np.array([S.NEAR_SLOT[0], S.NEAR_SLOT[1], 0.0])
    print(f"周回{rnd}: 床に{len(floor)}個 / ER 2: 残り{ans.get('remaining_on_floor')} pick={pick_yx} ({ans.get('pick_color')})"
          f" -> 最寄り '{tgt}' ずれ {err*1000:.0f}mm / place -> ({place_w[0]:.2f}, {place_w[1]:.2f})")
    marks = [(pick_yx, "ER 2: pick", RED), (cam.project(place_w), "ER 2: place", BLUE)]
    lines = [(f"Round {rnd}   on floor: {len(floor)}", WHITE),
             (f"ER 2: pick {ans.get('pick_color') or '?'} -> box", RED)]
    run_motion(S.block_xyz(scene, tgt), place_w[:2], marks, lines)
    S.settle(scene, 0.5)
    ok = S.in_box(scene, tgt)
    entry.update({"target": tgt, "pick_err_mm": round(err * 1000), "place": place_w[:2].tolist(), "result": "in box" if ok else "missed"})
    print(f"周回{rnd}: '{tgt}' -> {'箱に入った' if ok else '入らなかった'}")
    if not args.no_render:
        for _ in range(15):
            save(draw_frame([], [(f"Round {rnd} done", WHITE), (f"{tgt}: {'in box' if ok else 'missed'}", GREEN if ok else RED),
                                 (f"left on floor: {len([c for c, _ in S.LAYOUT if not S.in_box(scene, c)])}", WHITE)]))
            scene.step(S.DT)

rest = [c for c, _ in S.LAYOUT if not S.in_box(scene, c)]
summary = {"rounds": len(log), "in_box": [c for c, _ in S.LAYOUT if S.in_box(scene, c)], "left_on_floor": rest, "frames": saved[0]}
if not args.no_render:
    for _ in range(30):
        save(draw_frame([], [("Done" if not rest else "Stopped", WHITE), (f"in box: {4 - len(rest)} / 4", GREEN if not rest else WHITE)]))
        scene.step(S.DT)
S.dump({"log": log, "summary": summary}, os.path.join(args.out, "loop_log.json"))
print("SUMMARY", json.dumps(summary, ensure_ascii=False))
sys.stdout.flush(); os._exit(0)

In [ ]:
# アーム＋グリッパ用の位置制御パラメータ（SuperDex に同梱されていないので自作したもの）
import json
json.dump({"poseControllerParams": {"jointTracking": [{"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 35.64129638671875, "saturation": -1, "stiffness": 559.8521728515625}, {"damping": 103.19426727294922, "saturation": -1, "stiffness": 1620.9718017578125}, {"damping": 62.774349212646484, "saturation": -1, "stiffness": 986.0571899414062}, {"damping": 57.7801628112793, "saturation": -1, "stiffness": 907.6087036132812}, {"damping": 7.8675618171691895, "saturation": -1, "stiffness": 123.5833740234375}, {"damping": 8.216931343078613, "saturation": -1, "stiffness": 129.07125854492188}, {"damping": 7.02333927154541, "saturation": -1, "stiffness": 110.32235717773438}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 2, "saturation": -1, "stiffness": 20}, {"damping": 30, "saturation": -1, "stiffness": 500}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 2, "saturation": -1, "stiffness": 20}, {"damping": 2, "saturation": -1, "stiffness": 20}, {"damping": 30, "saturation": -1, "stiffness": 500}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 2, "saturation": -1, "stiffness": 20}], "linkPosTracking": [{"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}], "linkRotTracking": [{"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}, {"damping": 0, "saturation": -1, "stiffness": 0}]}}, open('/content/fr3_v2_2f_85_pose.superdex_controller', 'w'))
print('ok')

## 手順1 ── 場面を作って、写真を撮る

床にブロックが4個、右に仕切り付きの箱、左にロボットアーム。
シミュレータなので、カメラの位置・向き・画角と、ブロックの「本当の位置」も同時に書き出せます。

In [ ]:
%%bash
source /content/py.env
cd /content/project_superdex
SUPERDEX_ASSETS_PATH=$PWD/assets PYTHONPATH=/content $PY /content/sim_scene.py /content/step1 2>&1 | grep -v "^\[polyscope\]"

In [ ]:
from IPython.display import Image, display
import json
display(Image('/content/step1/scene.png', width=800))
truth = json.load(open('/content/step1/truth.json'))
print('本当の位置（重心・m）:')
for c, p in truth['blocks'].items():
    print(f"  {c:7s} x={p[0]:.3f} y={p[1]:.3f} z={p[2]:.4f}")
print('箱の範囲: x', [round(v, 2) for v in (truth['box']['min'][0], truth['box']['max'][0])],
      ' y', [round(v, 2) for v in (truth['box']['min'][1], truth['box']['max'][1])])

## 手順2 ── ER 2 に「床のブロックを指して」

Gemini Robotics ER 2 は、写真の上の位置を `[y, x]`（0〜1000）で返すモデルです。
聞き方のコツが2つあります。

- **個数を指定しない**。「4個ある」と言うと、見つけていない分を埋めてくる
- **無ければ空で返す**逃げ道を用意する

返ってきた点は、必ず写真の上に描いて確かめます（数字のままでは合っているか分からない）。

In [ ]:
import base64, json, re, requests
from PIL import Image as PILImage, ImageDraw

def get_key():
    '''左メニューの 🔑 シークレットに登録した GEMINI_API_KEY を読む（環境変数のようなもの）。'''
    from google.colab import userdata
    k = None
    for _ in range(3):                       # 許可ダイアログの直後は内部の応答が混ざることがあるので、文字列になるまで読み直す
        try:
            k = userdata.get('GEMINI_API_KEY')
        except Exception:
            k = None
        if isinstance(k, str) and k.strip():
            break
        k = None
    if not k:
        raise SystemExit("GEMINI_API_KEY が読めません。左メニューの 🔑（シークレット）に GEMINI_API_KEY を登録し、"
                         "「ノートブックからのアクセス」を ON にしてから、このセルを実行し直してください。")
    return k

API_KEY = get_key()
ER2_URL = "https://generativelanguage.googleapis.com/v1beta/interactions"

def ask_er2(image_path, prompt, level="low"):
    '''写真と質問を ER 2 に送り、返ってきた文章を返す。'''
    data = base64.b64encode(open(image_path, 'rb').read()).decode()
    body = {"model": "gemini-robotics-er-2-preview",
            "input": [{"type": "image", "data": data, "mime_type": "image/png"},
                      {"type": "text", "text": prompt}],
            "generation_config": {"thinking_level": level}}
    r = requests.post(ER2_URL, headers={"x-goog-api-key": API_KEY, "Content-Type": "application/json"},
                      json=body, timeout=180)
    j = r.json()
    if isinstance(j, dict) and "error" in j:
        raise RuntimeError(j["error"])
    # 応答の JSON をたどって、モデルが書いた文章（"text"）を全部つなぐ
    texts = []
    def walk(o):
        if isinstance(o, dict):
            if isinstance(o.get("text"), str):
                texts.append(o["text"])
            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for v in o:
                walk(v)
    walk(j)
    if not texts:
        raise RuntimeError(f"ER 2 から文章が返りませんでした（HTTP {r.status_code}）: {r.text[:800]}")
    return "\n".join(texts)

def parse_json(text):
    '''返答の中の JSON を取り出す（```json ... ``` で囲まれていてもよい）。'''
    m = re.search(r"\{.*\}|\[.*\]", text, re.S)
    return json.loads(m.group(0)) if m else None

PROMPT = '''これはロボットアームと、床に置かれた色つきブロック、黄色い仕切り付きの箱を写したシミュレーション画像です。
次の JSON だけを返してください。
{
  "blocks": [ {"color": "<見えたままの色>", "point": [y, x]}, ... ],   // 床に置かれているブロックを指す点。箱の中のものは除く。見つかったものだけ。無ければ []
  "place": [y, x]                                                       // 箱の中で、ロボットアームに近い側の区画の、空いている場所を指す点。無ければ null
}
個数は指定しません。座標は [y, x] で 0〜1000 です。'''

answer_text = ask_er2('/content/step1/scene.png', PROMPT)
answer = parse_json(answer_text)
if not isinstance(answer, dict):
    raise RuntimeError("返答を JSON として読めませんでした。返答そのもの:\n" + answer_text[:800])
print(json.dumps(answer, ensure_ascii=False, indent=1))

In [ ]:
# 返ってきた点を写真に描いて確かめる
img = PILImage.open('/content/step1/scene.png').convert('RGB')
d = ImageDraw.Draw(img)
sx, sy = img.size[0] / 1000, img.size[1] / 1000
def mark(yx, col, label):
    px, py = int(yx[1] * sx), int(yx[0] * sy)
    d.ellipse([px - 18, py - 18, px + 18, py + 18], outline=col, width=5)
    d.text((px + 22, py - 10), label, fill=col)
for i, b in enumerate(answer.get('blocks', [])):
    mark(b['point'], (255, 60, 60), f"{i}:{b.get('color')}")
if answer.get('place'):
    mark(answer['place'], (60, 140, 255), 'place')
img.save('/content/step2_marked.png')
display(Image('/content/step2_marked.png', width=800))
print(len(answer.get('blocks', [])), '個を指した')

## 手順3 ── 写真の点を、ロボットの座標に直す

写真の1点は、奥行きが分からないので「カメラから伸びる1本の線」にしかなりません。
その線が**高さの分かっている面（床のブロックの中心の高さ）**にぶつかる場所を求めると、3次元の座標になります。

1. `[y, x]`（0〜1000）を、中心が 0・端が ±1 になるように直す（ndc）
2. カメラの前・右・上のベクトルと画角から、その点を通る**光線の向き** $\mathbf{d}$ を作る
3. 光線と平面 $z = z_0$ の交点 $\mathbf{X} = \mathbf{p} + t\,\mathbf{d}$ を求める（$t = (z_0 - p_z)/d_z$）

シミュレータならカメラの情報は全部分かっています。実機では、カメラの校正か深度センサが要ります。ここが実機化の壁です。

置く場所は ER 2 が指した点を使いますが、**アームが届く範囲か**は仕組みの側で確かめます（箱の奥の区画は根元から約 1 m で、届く範囲の端）。遠ければロボットに近い側の区画に寄せます。

In [ ]:
import numpy as np
cam = json.load(open('/content/step1/camera.json'))
p, f, r, u = (np.array(cam[k]) for k in ('pos', 'f', 'r', 'u'))
tan_half = np.tan(np.deg2rad(cam['fov_deg']) / 2)
aspect = cam['aspect']

def unproject(yx, z0=cam['z_block']):
    '''写真の点 [y, x]（0〜1000）→ 平面 z=z0 との交点（ワールド座標）。床と交わらなければ None。'''
    y, x = yx
    nx, ny = x / 500 - 1, 1 - y / 500                    # 1. ndc
    d = f + nx * tan_half * aspect * r + ny * tan_half * u   # 2. 光線の向き
    if d[2] > -1e-6:
        return None
    t = (z0 - p[2]) / d[2]                               # 3. 平面との交点
    return p + t * d

truth = json.load(open('/content/step1/truth.json'))
blocks_true = {c: np.array(v) for c, v in truth['blocks'].items()}

print('ER 2 の点 → ロボットの座標 → いちばん近い本当のブロック')
picks = []
for b in answer.get('blocks', []):
    X = unproject(b['point'])
    if X is None:
        print(f"  {b['point']} は床と交わらない"); continue
    near = min(blocks_true, key=lambda c: np.linalg.norm(blocks_true[c][:2] - X[:2]))
    err = np.linalg.norm(blocks_true[near][:2] - X[:2]) * 1000
    picks.append((near, X, b['point']))
    print(f"  {str(b['point']):12s} ({b.get('color'):7s}) → ({X[0]:.3f}, {X[1]:.3f})  最寄り '{near}'  ずれ {err:.0f} mm")

place_w = unproject(answer['place']) if answer.get('place') else None
bmin, bmax = np.array(truth['box']['min']), np.array(truth['box']['max'])
REACH = 0.80                                   # アームが楽に届く距離（根元から・m）。FR3 の最大は約 0.85 m
NEAR_SLOT = np.array([0.54, 0.54, 0.0])        # 箱の、ロボットに近い側の区画の中ほど
in_box = place_w is not None and bmin[0] <= place_w[0] <= bmax[0] and bmin[1] <= place_w[1] <= bmax[1]
if not in_box:
    print('置き場所が箱の外だったので、近い側の区画に寄せます')
    place_w = NEAR_SLOT
elif np.linalg.norm(place_w[:2]) > REACH:
    print(f'置き場所が根元から {np.linalg.norm(place_w[:2]):.2f} m と遠いので、近い側の区画に寄せます')
    place_w = NEAR_SLOT
print(f'置く場所: ({place_w[0]:.3f}, {place_w[1]:.3f})  根元からの距離 {np.linalg.norm(place_w[:2]):.2f} m')

## 手順4 ── ロボットに指示して、掴んで箱へ

ER 2 が決めたのは「どれ」だけ。掴む位置はシミュレータの本当の重心を使い、
手先をそこへ持っていく関節の角度は IK（逆運動学）で解き、位置制御で追わせます。

最初に指した1個を掴みます。1〜2 分かかります。

In [ ]:
import subprocess, os
target, X, yx = picks[0]
cmd = ['/content/venv/bin/python', '/content/sim_pick.py', '--target', target,
       '--place', f'{place_w[0]:.4f}', f'{place_w[1]:.4f}', '--out', '/content/step4',
       '--pick-yx', str(yx[0]), str(yx[1])]
if answer.get('place'):
    cmd += ['--place-yx', str(answer['place'][0]), str(answer['place'][1])]
env = dict(os.environ, SUPERDEX_ASSETS_PATH='/content/project_superdex/assets', PYTHONPATH='/content')
print('掴む:', target, ' 置く:', [round(float(v), 3) for v in place_w[:2]])
out = subprocess.run(cmd, cwd='/content/project_superdex', env=env, capture_output=True, text=True)
print('\n'.join(l for l in out.stdout.splitlines() if not l.startswith('[polyscope]'))[-800:])
if out.returncode != 0:
    print(out.stderr[-1500:])
result = json.load(open('/content/step4/result.json'))
print('箱に入った' if result['in_box'] else '入らなかった', '/ 持ち上げ', result['lifted_mm'], 'mm')

In [ ]:
# コマを動画にして見る
!ffmpeg -y -loglevel error -framerate 25 -i /content/step4/f_%04d.png -c:v libx264 -pix_fmt yuv420p -vf "scale=960:-2" /content/step4.mp4
from IPython.display import Video
Video('/content/step4.mp4', embed=True, width=800)

## 手順5 ── 床が空になるまで、ER 2 に回させる

ここまでは1個だけでした。今度は**目標（床にブロックが残っていない）を達成するまで**、
写真を撮る → ER 2 に聞く → 座標に直す → 掴んで箱へ、を ER 2 に繰り返させます。

毎周回、ER 2 には「床に何個残っているか」「次に掴むのはどれか」「箱のどこへ置くか」だけを聞きます。
失敗しても、次の周回で ER 2 が「まだ残っている」と見て指示し直します。判定は本当の位置で行い、失敗の原因は ER 2 に聞きません。

場面を保ったまま繰り返す必要があるので、手順2〜4 を1つのスクリプト（`sim_loop.py`）にまとめてあります。
API キーはシークレットから環境変数で渡します。5〜8 分かかります。

In [ ]:
import subprocess, os, json
env = dict(os.environ, SUPERDEX_ASSETS_PATH='/content/project_superdex/assets', PYTHONPATH='/content',
           GEMINI_API_KEY=API_KEY)
proc = subprocess.Popen(['/content/venv/bin/python', '/content/sim_loop.py', '--out', '/content/step5', '--max-rounds', '6'],
                        cwd='/content/project_superdex', env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if not line.startswith('[polyscope]'):
        print(line.rstrip())
proc.wait()
log = json.load(open('/content/step5/loop_log.json'))
print()
print('周回 | ER 2 が言った残り | 床の本当の残り | 掴んだ | ずれ | 結果')
for e in log['log']:
    er = e.get('er2') or {}
    print(f"{e['round']:>4} | {str(er.get('remaining_on_floor')):>17} | {e['floor_truth']:>14} | {e.get('target','-'):>6} | {str(e.get('pick_err_mm','-')):>3} mm | {e.get('result')}")
print('箱の中:', log['summary']['in_box'], ' 床に残り:', log['summary']['left_on_floor'])

In [ ]:
!ffmpeg -y -loglevel error -framerate 25 -i /content/step5/f_%05d.png -c:v libx264 -pix_fmt yuv420p -vf "scale=960:-2" /content/step5.mp4
from IPython.display import Video
Video('/content/step5.mp4', embed=True, width=800)

## まとめ

| 決めること | 誰が | どうやって |
|---|---|---|
| どれを掴むか・どこへ置くか | ER 2 | 写真 → `[y, x]` |
| どこにあるか | シミュレータ | 写真の点を光線にして床と交わらせる。掴む位置は本当の重心 |
| 関節をどう曲げるか | IK ソルバ | 手先の位置から関節角を逆算 |
| 姿勢を保つ・握る | 位置制御 | 解いた関節角を追わせる |
| どこまで進んだか・続けるか | ER 2 | 毎周回、写真を見て残りの個数を答える。仕組みの側は本当の位置で判定して止める |

「指せる」と「動かせる」の間には、座標変換・関節角の計算・制御があります。
AI が決めるのは**方針（どれ・どこ）**で、値と実行は仕組みの側が受け持ちます。

## やってみる

- 手順4で `picks[0]` を `picks[1]` に変えて、別のブロックを掴む
- 手順3の `place_w` を手で書き換えて、箱の別の区画に置く（箱の範囲は手順1に出ています）
- 手順2のプロンプトを変えて、返ってくる点がどう変わるか見る（「一番手前のブロックだけ」など）
- 手順5の周回の上限（`--max-rounds`）や、ER 2 に聞く文面（`sim_loop.py` の `PROMPT`）を変えて、動きがどう変わるか見る
- 手順5で ER 2 に「失敗の原因」を聞くとどうなるか試す（動画に写っていないことを聞いたとき、何が返るか）
- 実機でやるなら、手順3の「カメラの情報」はどうやって手に入れるか考える

## 出典・ライセンス

- SuperDex: Meta Platforms, Inc.（コード Apache-2.0・アセット CC-BY-4.0）。FR3 V2 アーム（Franka Robotics・Apache-2.0）、Robotiq 2F-85（BSD-2-Clause）、Box-and-Blocks プレハブ（Meta Platforms, Inc.・CC-BY-4.0）
- Gemini Robotics ER 2: Google DeepMind
- このノートブックとスクリプト: MIT License（株式会社PROMPT-X）